# 02.5 - Greedy Algorithms

## What is greedy?
At each step, take the choice that looks **best right now**, and never go back.

Greedy is fast and simple - but **only correct for some problems**. It works when:
1. **Greedy-choice property:** a locally best choice is part of some globally best solution.
2. **Optimal substructure:** after the choice, the rest is a smaller problem of the same kind.

When greedy fails → usually **dynamic programming** (02.7).

## Where is it used?
| Real system | Greedy idea |
|---|---|
| **Huffman coding** (ZIP, JPEG, MP3) | Merge the two least frequent symbols first |
| **Dijkstra & Prim** | Always extend with the closest node / cheapest edge |
| **Scheduling** (meeting rooms, CPU, deliveries) | Pick the task that finishes earliest |
| **Making change** with standard coins/notes | Use the biggest note first |
| **Load balancing** | Send the job to the least-loaded server |
| **Caching** | Evict the item used farthest in the future / least recently |

## 1. When greedy works - and when it fails (coin change)

In [1]:
def greedy_change(coins, amount):
    used = []
    for c in sorted(coins, reverse=True):
        while amount >= c:
            amount -= c
            used.append(c)
    return used if amount == 0 else None

# Standard currency: greedy is optimal
print("RWF notes/coins, 3,750:", greedy_change([5000, 2000, 1000, 500, 100, 50, 20, 10, 5, 1], 3750))

# Weird coin system: greedy is WRONG
print("coins [1, 3, 4], amount 6 -> greedy:", greedy_change([1, 3, 4], 6), "but optimal is [3, 3]")
# -> for this, use DP (Coin Change, 02.7 / 03.1)

RWF notes/coins, 3,750: [2000, 1000, 500, 100, 100, 50]
coins [1, 3, 4], amount 6 -> greedy: [4, 1, 1] but optimal is [3, 3]


## 2. Classic: Activity selection (maximum non-overlapping meetings)
**Greedy rule:** sort by **end time**, always pick the meeting that finishes earliest.

In [2]:
def max_meetings(meetings):
    chosen, last_end = [], float("-inf")
    for start, end in sorted(meetings, key=lambda m: m[1]):
        if start >= last_end:
            chosen.append((start, end))
            last_end = end
    return chosen

meetings = [(9, 10), (9, 12), (10, 11), (11, 13), (12, 14), (13, 15)]
print(max_meetings(meetings))

[(9, 10), (10, 11), (11, 13), (13, 15)]


## 3. Real-world: Huffman coding (how compression works)
Frequent characters get **short** bit codes, rare characters get **long** ones.
Greedy: repeatedly merge the **two least frequent** nodes (heap).

In [3]:
import heapq
from collections import Counter
from itertools import count

def huffman_codes(text):
    tiebreak = count()
    heap = [(freq, next(tiebreak), ch) for ch, freq in Counter(text).items()]
    heapq.heapify(heap)
    if len(heap) == 1:
        return {heap[0][2]: "0"}
    while len(heap) > 1:
        f1, _, left = heapq.heappop(heap)
        f2, _, right = heapq.heappop(heap)
        heapq.heappush(heap, (f1 + f2, next(tiebreak), (left, right)))
    codes = {}
    def walk(node, code):
        if isinstance(node, str):
            codes[node] = code
        else:
            walk(node[0], code + "0")
            walk(node[1], code + "1")
    walk(heap[0][2], "")
    return codes

text = "abracadabra alakazam"
codes = huffman_codes(text)
for ch, code in sorted(codes.items(), key=lambda kv: len(kv[1])):
    print(repr(ch), code)
encoded_bits = sum(len(codes[ch]) for ch in text)
print(f"fixed 8-bit: {8 * len(text)} bits  ->  Huffman: {encoded_bits} bits")

'a' 0
' ' 1000
'l' 1001
'k' 1010
'z' 1011
'm' 1100
'b' 1101
'r' 1110
'c' 11110
'd' 11111
fixed 8-bit: 160 bits  ->  Huffman: 55 bits


---
## Solved problems

### 1. Assign Cookies - LC #455 (Easy)
Child `i` is content with a cookie of size ≥ `g[i]`. Max number of content children.

**Idea:** Sort both; give each cookie (smallest first) to the least greedy child it can satisfy.

In [4]:
def find_content_children(g, s):
    g.sort(); s.sort()
    child = 0
    for cookie in s:
        if child < len(g) and cookie >= g[child]:
            child += 1
    return child

assert find_content_children([1, 2, 3], [1, 1]) == 1
assert find_content_children([1, 2], [1, 2, 3]) == 2
print("Assign Cookies ")

Assign Cookies 


### 2. Jump Game - LC #55 (Medium) `nums[i]` = max jump length from `i`. Can you reach the last index?

**Idea:** Track the farthest index reachable so far; if you ever stand beyond it, you're stuck.

In [5]:
def can_jump(nums):
    reach = 0
    for i, jump in enumerate(nums):
        if i > reach:
            return False                 # stuck before reaching i
        reach = max(reach, i + jump)
    return True

assert can_jump([2, 3, 1, 1, 4])
assert not can_jump([3, 2, 1, 0, 4])
print("Jump Game ")

Jump Game 


### 3. Jump Game II - LC #45 (Medium)
Minimum number of jumps to reach the end (always possible).

**Idea:** BFS-like levels: each 'level' is the range reachable with one more jump.

In [6]:
def jump(nums):
    jumps, current_end, farthest = 0, 0, 0
    for i in range(len(nums) - 1):
        farthest = max(farthest, i + nums[i])
        if i == current_end:              # must jump now
            jumps += 1
            current_end = farthest
    return jumps

assert jump([2, 3, 1, 1, 4]) == 2
assert jump([2, 3, 0, 1, 4]) == 2
assert jump([0]) == 0
print("Jump Game II ")

Jump Game II 


### 4. Gas Station - LC #134 (Medium)
Circular route; start station that lets you complete the loop, or -1.

**Idea:** If total gas ≥ total cost a solution exists. Whenever the tank goes negative, no station between start and here can be the start → restart after it.

In [7]:
def can_complete_circuit(gas, cost):
    if sum(gas) < sum(cost):
        return -1
    tank, start = 0, 0
    for i in range(len(gas)):
        tank += gas[i] - cost[i]
        if tank < 0:                      # can't reach i+1 from start -> try from i+1
            start, tank = i + 1, 0
    return start

assert can_complete_circuit([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]) == 3
assert can_complete_circuit([2, 3, 4], [3, 4, 3]) == -1
print("Gas Station ")

Gas Station 


### 5. Non-overlapping Intervals - LC #435 (Medium)
Minimum intervals to remove so the rest don't overlap.

**Idea:** Keep the maximum set of non-overlapping intervals (sort by end); remove the rest.

In [8]:
def erase_overlap_intervals(intervals):
    intervals.sort(key=lambda x: x[1])      # earliest end first (activity selection)
    kept, last_end = 0, float("-inf")
    for start, end in intervals:
        if start >= last_end:
            kept += 1
            last_end = end
    return len(intervals) - kept

assert erase_overlap_intervals([[1, 2], [2, 3], [3, 4], [1, 3]]) == 1
assert erase_overlap_intervals([[1, 2], [1, 2], [1, 2]]) == 2
assert erase_overlap_intervals([[1, 2], [2, 3]]) == 0
print("Non-overlapping Intervals ")

Non-overlapping Intervals 


### 6. Partition Labels - LC #763 (Medium)
Split a string into as many parts as possible so each letter appears in at most one part.

**Idea:** Record each letter's last index; extend the current part until you reach the farthest last-index of the letters inside it.

In [9]:
def partition_labels(s):
    last = {ch: i for i, ch in enumerate(s)}
    sizes, start, end = [], 0, 0
    for i, ch in enumerate(s):
        end = max(end, last[ch])
        if i == end:
            sizes.append(end - start + 1)
            start = i + 1
    return sizes

assert partition_labels("ababcbacadefegdehijhklij") == [9, 7, 8]
assert partition_labels("eccbbbbdec") == [10]
print("Partition Labels ")

Partition Labels 


## More practice on LeetCode
Easy: Lemonade Change (#860), Best Time to Buy and Sell Stock II (#122)
Medium: Hand of Straights (#846), Merge Triplets (#1899), Minimum Number of Arrows (#452), Boats to Save People (#881)
Hard: Candy (#135), IPO (#502)

## Key takeaways
- Greedy = best local choice, never undo. Fast (often just a sort + one pass).
- **Prove it or find a counter-example** before trusting it (coin change [1,3,4] breaks greedy).
- Common greedy rules: sort by **end time**, take the **largest/smallest** first, track the **farthest reach**.

**Next: `02.6-Backtracking.ipynb`**